# Chapter 23: Capstone: The Full Tally Pipeline (notebook edition)

## Learning Objectives

- Assemble CI, build, verify and deploy stages with needs
- Review a workflow with a checklist run as code
- Read a green and a red run side by side
- Price a pipeline from measured job durations

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. The pipeline's shape

In [ ]:
from intro_gha import FIXTURES_DIR, WORKFLOWS_DIR
from intro_gha.workflow import execution_waves, job_graph, load_workflow

wf = load_workflow(WORKFLOWS_DIR / "ch23-pipeline.yml")
waves = execution_waves(job_graph(wf))
assert waves[0] == ["lint", "test"] and waves[-1] == ["summary"]
assert len(wf["jobs"]) == 7
print(waves)

## 2. The review checklist

In [ ]:
import json
from intro_gha.injection import injection_sites
from intro_gha.pinning import unpinned


def review(workflow):
    """Return findings from the capstone review checklist."""
    findings = []
    if "permissions" not in workflow:
        findings.append("no top-level permissions")
    for job_id, job in workflow["jobs"].items():
        if "timeout-minutes" not in job:
            findings.append(f"{job_id}: no timeout-minutes")
    findings += [f"unpinned: {r}" for r in unpinned(workflow)]
    findings += [f"injection: {s}" for s in injection_sites(workflow)]
    return findings


assert review(wf) == []
bad = json.loads(json.dumps(wf))
del bad["permissions"]
del bad["jobs"]["lint"]["timeout-minutes"]
assert review(bad) == ["no top-level permissions", "lint: no timeout-minutes"]
writers = [j for j, s in wf["jobs"].items() if "id-token" in s.get("permissions", {})]
assert writers == ["build"]
print("clean, and the checklist can fail")

## 3. The green and red runs

In [ ]:
S = json.loads((FIXTURES_DIR / "capstone_ch23_summary.json").read_text())
g, r = S["green"], S["red"]
assert g["conclusion"] == "success" and g["skipped"] == ["deploy_production"]
assert g["verify"]["attestation"] == "verified"
assert r["failed"] == ["test (3.13)"]
assert r["skipped"] == ["build", "verify", "deploy_staging", "deploy_production"]
assert r["summary"]["test"] == "failure"
print("green:", g["summary"])
print("red:  ", r["summary"])

## 4. Cost from measured durations

In [ ]:
from intro_gha.cost import mills_to_dollars, run_cost_mills, run_minutes

green = [(s, "linux") for s in g["job_seconds"].values()]
red = [(s, "linux") for s in r["job_seconds"].values()]
assert (run_minutes(green), run_cost_mills(green)) == (8, 48)
assert (run_minutes(red), run_cost_mills(red)) == (5, 30)
one_job = [(sum(g["job_seconds"].values()), "linux")]
assert run_minutes(one_job) == 3
print(mills_to_dollars(run_cost_mills(green)), "vs", mills_to_dollars(run_cost_mills(one_job)))

## Chapter Link

Read: `learning_modules/chapter_23_capstone.md`